# KAT-VAD v2 — **TAD clip-level gate** `s = c(v) · minmax_v(y_A3ign)` (Amendment 16, §27 G1–G10)

Pre-registration: `core/docs/v2/PREREG_ADDENDUM.md` **§27**. §26 showed A3-ign blind at clip level on TAD
(clip-level AUC 0.544). Here a **length-blind logistic gate** on the mean raw CLIP feature is fitted on
**MSAD-train** (never on TAD), checked once on **TAD-train** (G5 dev gate, bar 0.60), and only then multiplied into
§26's saved A3-ign scores on TAD-test. Macro is unchanged by construction (G3); the endpoint is micro (G6).
**No v2 model is trained or re-scored. `B` stays A3-ign.**

| step | what | GPU? |
|---|---|---|
| 0 | setup; §27 must be in the upload | — |
| 1 | preflight: MSAD-train 120/360, `MSAD_ncc`, TAD test ids, §26's score files | — |
| 2 | TAD-train features: from `TAD_ncc` if present, else extracted from the Kaggle frames (fallback) | only in the fallback |
| 3 | fit the gate on MSAD-train | — |
| 4 | score TAD-train, TAD-test, MSAD-test | — |
| 5 | read-out: dev gate first; the test only if it passes | — |

Runtime: a CPU runtime is enough unless step 2 falls back to extraction (then pick a GPU).

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import sys
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read: data/{MSAD,TAD}, cache/clip/{MSAD_ncc,TAD_ncc}
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # read: kat-vad/, outputs/v2_tadz; write: outputs/v2_tadg (+ fallback cache)
os.environ['PROJECT_ROOT']       = DRIVE_V2
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_V2}/outputs'

os.environ['REPO'] = f'{DRIVE_V2}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v2 there'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'
assert (REPO / 'core/tools/tad_clip_gate.py').is_file(), 'this upload predates §27 -- re-upload branch v2'
addendum = (REPO / 'core/docs/v2/PREREG_ADDENDUM.md').read_text(encoding='utf-8')
assert '## 27. Amendment 16' in addendum, 'upload has no addendum §27'
assert 'Authorized by the user on' in addendum.split('## 27. Amendment 16', 1)[1], '§27 is not signed'

STRIDE   = constants.V2_TADZ_STRIDE                               # 8 (§26's protocol)
MSAD     = constants.DATA_ROOT / 'MSAD'                           # Thesis/data/MSAD
MSAD_NCC = constants.CLIP_CACHE_DIR / 'MSAD_ncc'                  # same extractor as TAD_ncc (G4, C2)
TAD      = constants.DATA_ROOT / constants.TAD_DATASET            # Thesis/data/TAD
TAD_NCC  = constants.CLIP_CACHE_DIR / 'TAD_ncc'
TADZ     = constants.OUTPUT_ROOT / 'v2_tadz'                      # §26's score files (read only)
OUT      = constants.OUTPUT_ROOT / 'v2_tadg'
TRAIN_FALLBACK = Path(DRIVE_V2) / 'cache' / 'clip' / 'TAD_train_ncc'   # written only by the fallback
KAGGLE   = 'nikanvasei/traffic-anomaly-dataset-tad'
LOCAL    = Path('/content/tadg')
LOCAL.mkdir(parents=True, exist_ok=True)
for name in ('MSAD', 'MSAD_NCC', 'TAD', 'TAD_NCC', 'TADZ', 'OUT', 'TRAIN_FALLBACK'):
    print(f'{name:15s}', globals()[name])

In [ ]:
%%bash
# the readout helpers import core.data (av, faiss) and core.tools (transformers): same set as every v2 notebook
pip install -q "transformers==4.56.*" av einops faiss-cpu kagglehub
python -c "import av, faiss, transformers, sklearn, kagglehub; print('transformers', transformers.__version__, 'sklearn', sklearn.__version__)"
python -c "import core.tools.tad_clip_gate" && echo "tad_clip_gate imports OK"

In [ ]:
import json
import math
import re
import shutil
import subprocess
import time

import numpy as np

ENV = dict(os.environ)


def run(cmd, log=None):
    '''Stream a child's output here (and to ``log``); raise with its tail on failure.'''
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


def stage_ids(src, dst, ids):
    '''Copy ``ids`` (+ the dir's manifests) from Drive to VM disk once (bit-identical; C2).'''
    dst.mkdir(parents=True, exist_ok=True)
    for v in ids:
        if not (dst / f'{v}.npy').exists():
            shutil.copy2(src / f'{v}.npy', dst / f'{v}.npy')
    for m in list(src.glob('*.json')) + list(src.glob('*.npz')):
        shutil.copy2(m, dst / m.name)
    return dst


def publish(src, dst):
    '''Copy a VM-local result to Drive ATOMICALLY: ``<dst>.partial``, fsync, rename (lesson (bi)).'''
    if src.is_dir():
        for f in [p for p in src.rglob('*') if p.is_file()]:
            publish(f, dst / f.relative_to(src))
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + '.partial')
    shutil.copy2(src, tmp)
    with open(tmp, 'rb+') as fh:
        os.fsync(fh.fileno())
    os.replace(tmp, dst)


def complete(dir_, ids):
    return dir_.is_dir() and all((dir_ / f'{v}.npy').exists() for v in ids)

## 1. Preflight — every input is the one §27 names (G4, G9(a)(b))

In [ ]:
msad_train = json.loads((MSAD / constants.LABELS_TRAIN_FILENAME).read_text())
counts = (sum(msad_train.values()), len(msad_train) - sum(msad_train.values()))
assert counts == tuple(constants.V2_TADG_MSAD_TRAIN), f'G9(b): MSAD-train is {counts} abnormal/normal'
msad_test = json.loads((MSAD / constants.FRAME_LABELS_TEST_FILENAME).read_text())
for split, ids in (('train', msad_train), ('test', msad_test)):
    lost = [v for v in ids if not (MSAD_NCC / f'{v}.npy').exists()]
    assert not lost, f'MSAD_ncc lacks {len(lost)} {split} videos: {lost[:5]}'
print(f'MSAD-train {counts} abnormal/normal, MSAD-test {len(msad_test)}; all in MSAD_ncc')

labels_s8 = json.loads((TAD / constants.FRAME_LABELS_TEST_FILENAME).read_text())
test_ids = sorted(labels_s8)
assert len(test_ids) == constants.V2_TADZ_VIDEOS, len(test_ids)
for run_ in ['T0'] + [f'{constants.V2_TADG_ARM}/s{s}' for s in constants.V2_E3_SEEDS]:
    n = len(list((TADZ / run_ / 'scores').glob('*.npz')))
    assert n == len(test_ids), f'§26 run {run_} has {n} score files, expected {len(test_ids)}'
(LOCAL / 'test_ids.txt').write_text('\n'.join(test_ids) + '\n')
(LOCAL / 'msad_test_ids.txt').write_text('\n'.join(sorted(msad_test)) + '\n')
(LOCAL / 'msad_test_labels.json').write_text(json.dumps(msad_test))
print(f'TAD-test {len(test_ids)} videos; §26 score files present for T0 + A3-ign x 5')

## 2. TAD-train features (G5 dev gate, never fitted on)

TAD-train = every TAD video `tad_test_anno.json` does not name. The 2026-09 TAD campaign trained on it, so its
CLIP s8 features should already be in `TAD_ncc`. If fewer than 300 are there, this cell falls back to extracting
them from the Kaggle frames (numeric-order symlink farm, as §26) into `Thesis-V2/cache/clip/TAD_train_ncc` —
**needs a GPU runtime**. Labels: the Kaggle top-level folder (`abnormal/` / `normal/`) when the farm is built,
else `labels_train.json`, else the id prefix; the prefix rule is checked on the 100 test ids by the read-out (G9(c)).

In [ ]:
def by_prefix(ids):
    return {v: int(not v.startswith(constants.V2_TADG_NORMAL_PREFIX)) for v in ids}


assert by_prefix(test_ids) == {v: int(max(g) > 0) for v, g in labels_s8.items()}, 'G9(c): prefix rule fails on test'
dev_ids = sorted(p.stem for p in TAD_NCC.glob('*.npy') if p.stem not in set(test_ids))
DEV_DIR = TAD_NCC
if len(dev_ids) >= constants.V2_TADG_DEV_MIN_VIDEOS:
    lt = json.loads((TAD / constants.LABELS_TRAIN_FILENAME).read_text()) if (TAD / constants.LABELS_TRAIN_FILENAME).exists() else {}
    dev_labels = {v: int(lt[v]) for v in dev_ids} if lt and set(dev_ids) <= set(lt) else by_prefix(dev_ids)
    src = 'labels_train.json' if lt and set(dev_ids) <= set(lt) else 'id prefix'
    print(f'TAD-train from TAD_ncc: {len(dev_ids)} videos, labels from {src}')
else:
    import kagglehub
    print(f'TAD_ncc has only {len(dev_ids)} non-test videos -> fallback extraction from Kaggle')
    root = Path(kagglehub.dataset_download(KAGGLE))
    FARM, IMG, found = Path('/content/tadg_farm'), {'.jpg', '.jpeg', '.png', '.bmp'}, {}
    for dirpath, _, files in os.walk(root):
        if not any(Path(f).suffix.lower() in IMG for f in files):
            continue
        d = Path(dirpath)
        vid = re.sub(r'\.(mp4|avi|mov|mkv)$', '', d.name, flags=re.I)
        cls = next((p.name for p in d.parents if p.name in (constants.TAD_ABNORMAL_DIRNAME, constants.TAD_NORMAL_DIRNAME)), None)
        if vid in set(test_ids) or cls is None:
            continue
        assert vid not in found, f'{vid} appears twice'
        found[vid] = (d, int(cls == constants.TAD_ABNORMAL_DIRNAME))

    def key(p):
        return int(re.findall(r'\d+', p.stem)[-1])

    if FARM.exists():
        shutil.rmtree(FARM)
    for vid, (d, _) in sorted(found.items()):
        frames = sorted((p for p in d.iterdir() if p.suffix.lower() in IMG), key=key)
        (FARM / vid).mkdir(parents=True)
        for i, p in enumerate(frames):
            (FARM / vid / f'{i:06d}{p.suffix.lower()}').symlink_to(p)
    dev_ids = sorted(found)
    dev_labels = {v: found[v][1] for v in dev_ids}
    (LOCAL / 'dev_ids.txt').write_text('\n'.join(dev_ids) + '\n')
    run([sys.executable, '-m', 'core.tools.extract_clip_features', '--frames-dir', FARM,
         '--ids-file', LOCAL / 'dev_ids.txt', '--dataset', constants.TAD_DATASET, '--stride', STRIDE,
         '--batch-size', 64, '--device', 'auto', '--no-center-crop', '--output-dir', LOCAL / 'dev_clip'],
        log=LOCAL / 'dev_clip.log')
    publish(LOCAL / 'dev_clip', TRAIN_FALLBACK)
    DEV_DIR = TRAIN_FALLBACK
    print(f'TAD-train extracted: {len(dev_ids)} videos, labels from the Kaggle folders')

n_abn = sum(dev_labels.values())
assert len(dev_ids) >= constants.V2_TADG_DEV_MIN_VIDEOS and 0 < n_abn < len(dev_ids), (len(dev_ids), n_abn)
(LOCAL / 'dev_ids.txt').write_text('\n'.join(dev_ids) + '\n')
(LOCAL / 'dev_labels.json').write_text(json.dumps(dev_labels))
print(f'dev: {len(dev_ids)} videos, {n_abn} abnormal')

## 3. Fit the gate on MSAD-train (G2, G4) — nothing swept

In [ ]:
L_MSAD = stage_ids(MSAD_NCC, LOCAL / 'stage' / 'MSAD_ncc', sorted(msad_train) + sorted(msad_test))
(LOCAL / 'msad_train_labels.json').write_text(json.dumps(msad_train))
GATE = LOCAL / 'gate.npz'
run([sys.executable, '-m', 'core.tools.tad_clip_gate', 'fit', '--clip-dir', L_MSAD,
     '--labels', LOCAL / 'msad_train_labels.json', '--out', GATE])
print(json.loads(GATE.with_suffix('.json').read_text()))

## 4. Score TAD-train, TAD-test, MSAD-test

In [ ]:
L_TEST = stage_ids(TAD_NCC, LOCAL / 'stage' / 'TAD_test', test_ids)
L_DEV = stage_ids(DEV_DIR, LOCAL / 'stage' / 'TAD_dev', dev_ids)
for name, clip_dir, ids in (('c_dev', L_DEV, 'dev_ids.txt'), ('c_test', L_TEST, 'test_ids.txt'),
                            ('c_msad_test', L_MSAD, 'msad_test_ids.txt')):
    run([sys.executable, '-m', 'core.tools.tad_clip_gate', 'score', '--gate', GATE, '--clip-dir', clip_dir,
         '--ids-file', LOCAL / ids, '--out', LOCAL / f'{name}.json'])

## 5. Read-out — the dev gate first; TAD-test only if it passes (G5–G7)

In [ ]:
run([sys.executable, '-m', 'core.tools.tad_clip_gate', 'readout', '--tadz-root', TADZ,
     '--dev', LOCAL / 'c_dev.json', '--dev-labels', LOCAL / 'dev_labels.json', '--test', LOCAL / 'c_test.json',
     '--msad-test', LOCAL / 'c_msad_test.json', '--msad-test-labels', LOCAL / 'msad_test_labels.json',
     '--out-dir', LOCAL / 'out'])
for f in ('gate.npz', 'gate.json', 'c_dev.json', 'c_test.json', 'c_msad_test.json', 'dev_labels.json'):
    publish(LOCAL / f, OUT / 'gate' / f)
publish(LOCAL / 'out', OUT)
print((OUT / 'REPORTS' / 'tadg_readout.md').read_text())

## 6. Record — then **stop**

Paste `v2_tadg/REPORTS/tadg_readout.md` back to Claude. The sentence is fixed by §27 G6; nothing (pooling, `C`,
features, corpus, fusion) is changed after reading it (G8). `B` = A3-ign stands; macro did not move (G3).